In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import cv2
import numpy as np
import pandas as pd
import datetime
import os

from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input

In [3]:
data_path = "/kaggle/input/datasets/jangedoo/utkface-new/UTKFace"

images = []
ages = []
genders = []

for img_name in os.listdir(data_path):
    try:
        age = int(img_name.split('_')[0])
        gender = int(img_name.split('_')[1])
        
        img = cv2.imread(os.path.join(data_path, img_name))
        
        if img is None:
            continue
        
        img = cv2.resize(img, (128, 128))
        
        images.append(img)
        ages.append(age)
        genders.append(gender)
        
    except:
        continue

X = np.array(images) / 255.0
y_age = np.array(ages)
y_gender = np.array(genders)

print("Dataset Loaded Successfully")
print("Total Images:", len(X))

Dataset Loaded Successfully
Total Images: 23708


In [4]:
input_layer = Input(shape=(128,128,3))

x = Conv2D(32, (3,3), activation='relu')(input_layer)
x = MaxPooling2D()(x)
x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D()(x)
x = Flatten()(x)

age_output = Dense(1, activation='linear', name='age')(x)
gender_output = Dense(1, activation='sigmoid', name='gender')(x)

model = Model(inputs=input_layer, outputs=[age_output, gender_output])

model.compile(
    optimizer='adam',
    loss={
        'age': 'mse',
        'gender': 'binary_crossentropy'
    },
    metrics={
        'age': 'mae',
        'gender': 'accuracy'
    }
)

model.summary()

I0000 00:00:1771423705.740982      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 126, 126,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 63, 63,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 61, 61,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 30, 30,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 57600)     │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │     57,601 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │     57,601 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 134,594 (525.76 KB)

 Trainable params: 134,594 (525.76 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.fit(
    X,
    {'age': y_age, 'gender': y_gender},
    epochs=10,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/10


I0000 00:00:1771423731.339703     126 service.cc:152] XLA service 0x79e968045340 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771423731.339736     126 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1771423731.737470     126 cuda_dnn.cc:529] Loaded cuDNN version 91002


 17/593 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - age_loss: 895.9880 - age_mae: 24.2445 - gender_accuracy: 0.4979 - gender_loss: 1.1119 - loss: 897.0999

I0000 00:00:1771423734.509145     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


593/593 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - age_loss: 408.5408 - age_mae: 15.3487 - gender_accuracy: 0.6950 - gender_loss: 0.6149 - loss: 409.1557 - val_age_loss: 218.3602 - val_age_mae: 11.8257 - val_gender_accuracy: 0.8518 - val_gender_loss: 0.3610 - val_loss: 218.6861
Epoch 2/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - age_loss: 193.1601 - age_mae: 10.7021 - gender_accuracy: 0.8400 - gender_loss: 0.3862 - loss: 193.5462 - val_age_loss: 166.5935 - val_age_mae: 9.7174 - val_gender_accuracy: 0.8726 - val_gender_loss: 0.3139 - val_loss: 166.8966
Epoch 3/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - age_loss: 167.6357 - age_mae: 9.9299 - gender_accuracy: 0.8567 - gender_loss: 0.3617 - loss: 167.9971 - val_age_loss: 174.2679 - val_age_mae: 9.8844 - val_gender_accuracy: 0.7402 - val_gender_loss: 0.7338 - val_loss: 175.0478
Epoch 4/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - age_loss: 143.2372 - age_mae: 9.1818 - gender_accuracy: 0.8658 - gender_loss: 0.3472 - loss: 143.5844 - val_a

In [6]:
model.save("age_gender_model.keras")


In [8]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)


In [ ]:
cap = cv2.VideoCapture("/kaggle/input/datasets/susrout/people-walking-videos/4750042-hd_1920_1080_30fps.mp4")

data_log = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        face = cv2.resize(face, (128,128))
        face = face / 255.0
        face = np.reshape(face, (1,128,128,3))
        
        age_pred, gender_pred = model.predict(face)
        
        age = int(age_pred[0][0])
        gender = "Male" if gender_pred[0][0] < 0.5 else "Female"
        
        senior = "Yes" if age > 60 else "No"
        
        current_time = datetime.datetime.now().strftime("%H:%M:%S")
        
        data_log.append([current_time, age, gender, senior])
        
        label = f"{age}, {gender}, Senior: {senior}"
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(frame, label, (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0,255,0), 2)

cap.release()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━

In [10]:
df = pd.DataFrame(data_log, columns=[
    "Time of Visit", "Age", "Gender", "Senior Citizen"
])

df.to_csv("visit_log.csv", index=False)

print("Data Saved Successfully")


Data Saved Successfully
